# JEDI Ridge -> XGBoost Stacked K-Fold Workbook

**Objective:** predict personal-fit scores with Ridge, append those predicted scores, then train XGBoost for `applied`.

**Input:** `data/jedi_master_prepared.csv`

**Main evaluation:** leakage-safe 5-fold CV.

## Step 0: Imports And Paths

In [14]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "xgboost is not installed in this kernel. Run '%pip install xgboost', restart the kernel, then rerun."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
N_SPLITS = 5


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "jedi_master_prepared.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/jedi_master_prepared.csv from this notebook location.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "jedi_master_prepared.csv"
OUTPUT_DIR = PROJECT_ROOT / "data"

print(PROJECT_ROOT)
print(DATA_PATH)

d:\ML\JEDI
d:\ML\JEDI\data\jedi_master_prepared.csv


## Step 1: Load Dataset

In [15]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head(3))
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

Rows: 691
Columns: 51


,row_id,source_platform,source_type,scraped_at,can_apply_now,company_name,company_match_score,role_title,role_title_preference_score,role_family,role_family_preference_score,seniority,seniority_profile_fit_score,location,location_profile_fit_score,work_mode,work_mode_fit_score,employment_type,experience_min_years,experience_max_years,experience_profile_fit_score,skills_or_requirements,skills_profile_fit_score,salary_min,salary_max,salary_profile_fit_score,freshness,overall_preference_score,preference_reason,applied,apply_url,skills_or_requirements_original,skills_array,has_skills_or_requirements,skills_imputation_type,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,is_bengaluru,is_gurgaon_ncr,is_india,experience_min_bucket,salary_midpoint,salary_available,near_duplicate_key,company_name_standard,role_title_standard
0,1,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Nirmata,4,"Software Engineer Intern, India",5,Backend / Cloud,4,intern,5,"Bangalore, Hybrid",4,Hybrid,5,internship,0.0,1.0,5.0,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,2,1200000,1200000,4,Not visible,3,"Intern role is fine, everything is good but the requirements state Kubernetes, observability tools, promethus and Go...",no,https://job-boards.greenhouse.io/nirmata/jobs/5040355008,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,"[""go"",""java"",""microservices"",""kubernetes"",""prometheus"",""timescale"",""reliability"",""automation""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,1200000,True,nirmata|software_engineer_intern_india|bengaluru,nirmata,software_engineer_intern_india
1,2,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,CloudSEK,5,SDE - Backend - Intern,5,Backend,5,intern,5,"Bengaluru, Karnataka, India",4,Hybrid,5,internship,0.0,1.0,5.0,Backend engineering; cybersecurity product; APIs; databases likely,4,800000,900000,3,Not visible,4,"Backend intern in Bengaluru; very relevant for fresher profile, salary is a little less and location bengaluru",yes,https://job-boards.greenhouse.io/cloudsek/jobs/5766402004,Backend engineering; cybersecurity product; APIs; databases likely,"[""backend engineering"",""cybersecurity product"",""apis"",""databases likely""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,850000,True,cloudsek|sde_backend_intern|bengaluru,cloudsek,sde_backend_intern
2,3,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Anaplan,5,Associate Software Engineer,5,Backend / Fullstack,5,entry_level,5,"Gurugram, India",5,unknown,5,full_time,0.0,2.0,5.0,Software engineering; AI-infused platform; SaaS; cloud; planning systems,5,500000,700000,0,Not visible,3,Associate level in Gurgaon; high-quality company and location fitbut salary is quite less and the bad factor,yes,https://job-boards.greenhouse.io/anaplan/jobs/8511489002,Software engineering; AI-infused platform; SaaS; cloud; planning systems,"[""software engineering"",""ai-infused platform"",""saas"",""cloud"",""planning systems""]",True,original,fullstack,entry_level,full_time,unknown,delhi_ncr,india,delhi_ncr,False,True,True,0,600000,True,anaplan|associate_software_engineer|delhi_ncr,anaplan,associate_software_engineer


,applied,rows
0,no,386
1,yes,305


## Step 2: Define Targets

In [16]:
SCORE_TARGETS = [
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "work_mode_fit_score",
    "experience_profile_fit_score",
    "skills_profile_fit_score",
    "salary_profile_fit_score",
    "overall_preference_score",
]

missing_score_targets = [c for c in SCORE_TARGETS if c not in df.columns]
if missing_score_targets:
    raise ValueError(f"Missing score target columns: {missing_score_targets}")

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
bad_applied = sorted(set(df["applied_clean"]) - {"yes", "no"})
if bad_applied:
    raise ValueError(f"Applied column has invalid values: {bad_applied}")

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)

for col in SCORE_TARGETS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_rows = df["target_applied"].notna() & df[SCORE_TARGETS].notna().all(axis=1)
model_df = df.loc[valid_rows].reset_index(drop=True).copy()

y_applied = model_df["target_applied"].astype(int)
y_scores = model_df[SCORE_TARGETS].astype(float)

score_audit = pd.DataFrame({
    "target": SCORE_TARGETS,
    "min": [y_scores[c].min() for c in SCORE_TARGETS],
    "max": [y_scores[c].max() for c in SCORE_TARGETS],
    "mean": [y_scores[c].mean() for c in SCORE_TARGETS],
    "missing": [model_df[c].isna().sum() for c in SCORE_TARGETS],
})

display(score_audit)
display(y_applied.value_counts().rename_axis("target_applied").reset_index(name="rows"))

,target,min,max,mean,missing
0,role_title_preference_score,0.0,5.0,4.201786,0
1,role_family_preference_score,1.0,5.0,4.180357,0
2,seniority_profile_fit_score,0.0,5.0,3.619643,0
3,location_profile_fit_score,0.0,5.0,4.060714,0
4,work_mode_fit_score,1.0,5.0,4.316071,0
5,experience_profile_fit_score,0.0,5.0,3.617857,0
6,skills_profile_fit_score,1.0,5.0,3.778571,0
7,salary_profile_fit_score,0.0,8.0,4.051786,0
8,overall_preference_score,0.0,5.0,3.416071,0


,target_applied,rows
0,1,288
1,0,272


## Step 3: Build Leakage-Free Raw Features

In [17]:
ALL_SCORE_COLUMNS = [
    c for c in model_df.columns
    if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_OR_METADATA_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_OR_METADATA_COLUMNS))
available_drop_columns = [c for c in drop_columns if c in model_df.columns]

meta_columns = [
    "row_id",
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "applied",
    "source_type",
]
meta = model_df[[c for c in meta_columns if c in model_df.columns]].copy()

candidate_raw_features = [c for c in model_df.columns if c not in available_drop_columns]

display(pd.DataFrame({"dropped_from_training_X": available_drop_columns}))
display(pd.DataFrame({"candidate_raw_feature": candidate_raw_features}))

,dropped_from_training_X
0,applied
1,applied_clean
2,apply_url
3,company_match_score
4,company_name
5,company_name_standard
6,experience_profile_fit_score
7,location_profile_fit_score
8,near_duplicate_key
9,overall_preference_score


,candidate_raw_feature
0,can_apply_now
1,role_title
2,role_family
3,seniority
4,location
5,work_mode
6,employment_type
7,experience_min_years
8,experience_max_years
9,skills_or_requirements


## Step 4: Feature Groups

In [18]:
text_source_columns = [
    "role_title",
    "role_family",
    "seniority",
    "location",
    "skills_or_requirements",
    "skills_or_requirements_original",
    "skills_array",
]

categorical_features = [
    "can_apply_now",
    "freshness",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
]

numeric_features = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

text_source_columns = [c for c in text_source_columns if c in model_df.columns]
categorical_features = [c for c in categorical_features if c in model_df.columns]
numeric_features = [c for c in numeric_features if c in model_df.columns]
boolean_features = [c for c in boolean_features if c in model_df.columns]

for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")


def to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return 1.0
    if text in {"false", "0", "no"}:
        return 0.0
    return np.nan


for col in boolean_features:
    model_df[col] = model_df[col].map(to_binary)

model_df["job_text"] = (
    model_df[text_source_columns]
    .fillna("")
    .astype(str)
    .agg(" | ".join, axis=1)
)

X_raw = model_df[candidate_raw_features].copy()
X_raw["job_text"] = model_df["job_text"]

display(pd.DataFrame({
    "feature_group": ["text_source", "categorical", "numeric", "boolean"],
    "columns": [
        ", ".join(text_source_columns),
        ", ".join(categorical_features),
        ", ".join(numeric_features),
        ", ".join(boolean_features),
    ],
}))

,feature_group,columns
0,text_source,"role_title, role_family, seniority, location, skills_or_requirements, skills_or_requirements_original, skills_array"
1,categorical,"can_apply_now, freshness, role_family_standard, seniority_standard, employment_type_standard, work_mode_standard, ci..."
2,numeric,"experience_min_years, salary_min, salary_max, salary_midpoint"
3,boolean,"has_skills_or_requirements, is_bengaluru, is_gurgaon_ncr, is_india, salary_available"


## Step 5: Pipelines

In [19]:
def make_preprocess(extra_numeric_features=None, text_max_features=1800):
    extra_numeric_features = list(extra_numeric_features or [])
    numeric_cols = numeric_features + boolean_features + extra_numeric_features

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])

    text_transformer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=text_max_features,
        sublinear_tf=True,
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_cols),
            ("categorical", categorical_transformer, categorical_features),
            ("text", text_transformer, "job_text"),
        ],
        remainder="drop",
    )


def make_score_model(alpha=2.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(text_max_features=2200)),
        ("regressor", Ridge(alpha=alpha)),
    ])


def make_xgb_model(extra_numeric_features=None, scale_pos_weight=1.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(extra_numeric_features=extra_numeric_features, text_max_features=1800)),
        ("classifier", XGBClassifier(
            n_estimators=350,
            max_depth=3,
            learning_rate=0.035,
            min_child_weight=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.10,
            reg_lambda=3.00,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
            scale_pos_weight=scale_pos_weight,
        )),
    ])


SCORE_BOUNDS = {
    "salary_profile_fit_score": (0.0, 8.0),
}


def clip_scores(predictions):
    predictions = np.asarray(predictions, dtype=float).copy()
    for idx, target in enumerate(SCORE_TARGETS):
        low, high = SCORE_BOUNDS.get(target, (0.0, 5.0))
        predictions[:, idx] = np.clip(predictions[:, idx], low, high)
    return predictions


PRED_SCORE_COLUMNS = [f"pred_{c}" for c in SCORE_TARGETS]
TRUE_SCORE_COLUMNS = [f"true_{c}" for c in SCORE_TARGETS]


def append_score_columns(X_base, score_array, prefix):
    X_aug = X_base.copy()
    score_array = clip_scores(score_array)
    for idx, score_name in enumerate(SCORE_TARGETS):
        X_aug[f"{prefix}_{score_name}"] = score_array[:, idx]
    return X_aug

## Step 6: Metrics

In [20]:
def regression_metrics(y_true, y_pred, split_name, fold=None):
    y_true_df = pd.DataFrame(np.asarray(y_true, dtype=float), columns=SCORE_TARGETS)
    y_pred_df = pd.DataFrame(clip_scores(y_pred), columns=SCORE_TARGETS)
    rows = []
    for target in SCORE_TARGETS:
        actual = y_true_df[target]
        pred = y_pred_df[target]
        abs_error = (pred - actual).abs()
        rows.append({
            "fold": fold,
            "split": split_name,
            "target": target,
            "mae": mean_absolute_error(actual, pred),
            "rmse": np.sqrt(np.mean(np.square(pred - actual))),
            "r2": r2_score(actual, pred),
            "spearman": actual.corr(pred, method="spearman"),
            "within_0_5": (abs_error <= 0.5).mean(),
            "within_1_0": (abs_error <= 1.0).mean(),
        })
    return pd.DataFrame(rows)


def classification_summary(model_name, fold, y_true, proba, threshold=0.50):
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    return {
        "model": model_name,
        "fold": fold,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }


def threshold_table(y_true, proba, model_name):
    rows = []
    for threshold in np.arange(0.20, 0.81, 0.05):
        pred = (np.asarray(proba) >= threshold).astype(int)
        rows.append({
            "model": model_name,
            "threshold": round(float(threshold), 2),
            "predicted_yes_count": int(pred.sum()),
            "precision_yes": precision_score(y_true, pred, zero_division=0),
            "recall_yes": recall_score(y_true, pred, zero_division=0),
            "f1_yes": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)

## Step 7: Leakage-Safe K-Fold Run

In [21]:
def run_stacked_xgb_cv():
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    classification_rows = []
    regression_rows = []
    prediction_frames = []

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_raw, y_applied), start=1):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_tr = y_applied.iloc[train_idx].copy()
        y_val = y_applied.iloc[val_idx].copy()
        y_tr_scores = y_scores.iloc[train_idx].copy()
        y_val_scores = y_scores.iloc[val_idx].copy()

        scale_pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

        fold_score_model = make_score_model(alpha=2.0)
        train_score_oof = cross_val_predict(
            fold_score_model,
            X_tr_raw,
            y_tr_scores,
            cv=inner_cv,
            n_jobs=None,
        )
        train_score_oof = clip_scores(train_score_oof)

        fold_score_model.fit(X_tr_raw, y_tr_scores)
        val_score_pred = clip_scores(fold_score_model.predict(X_val_raw))

        regression_rows.append(regression_metrics(y_tr_scores, train_score_oof, "train_inner_oof", fold=fold_idx))
        regression_rows.append(regression_metrics(y_val_scores, val_score_pred, "outer_validation", fold=fold_idx))

        X_tr_stacked = append_score_columns(X_tr_raw, train_score_oof, prefix="pred")
        X_val_stacked = append_score_columns(X_val_raw, val_score_pred, prefix="pred")
        X_tr_oracle = append_score_columns(X_tr_raw, y_tr_scores.values, prefix="true")
        X_val_oracle = append_score_columns(X_val_raw, y_val_scores.values, prefix="true")

        fold_configs = [
            ("raw_xgboost", make_xgb_model(scale_pos_weight=scale_pos_weight), X_tr_raw, X_val_raw),
            ("stacked_ridge_scores_xgboost", make_xgb_model(PRED_SCORE_COLUMNS, scale_pos_weight=scale_pos_weight), X_tr_stacked, X_val_stacked),
            ("oracle_true_scores_xgboost", make_xgb_model(TRUE_SCORE_COLUMNS, scale_pos_weight=scale_pos_weight), X_tr_oracle, X_val_oracle),
        ]

        fold_predictions = meta.iloc[val_idx].reset_index(drop=True).copy()
        fold_predictions["fold"] = fold_idx
        fold_predictions["actual_applied"] = y_val.reset_index(drop=True).values

        for target_idx, target in enumerate(SCORE_TARGETS):
            fold_predictions[f"true_{target}"] = y_val_scores.reset_index(drop=True)[target]
            fold_predictions[f"pred_{target}"] = val_score_pred[:, target_idx]

        for model_name, estimator, X_tr, X_val in fold_configs:
            estimator.fit(X_tr, y_tr)
            proba = estimator.predict_proba(X_val)[:, 1]
            classification_rows.append(classification_summary(model_name, fold_idx, y_val, proba, threshold=0.50))
            fold_predictions[f"{model_name}_probability"] = proba
            fold_predictions[f"{model_name}_prediction"] = (proba >= 0.50).astype(int)

        prediction_frames.append(fold_predictions)

    return (
        pd.DataFrame(classification_rows),
        pd.concat(regression_rows, ignore_index=True),
        pd.concat(prediction_frames, ignore_index=True).sort_values(["fold", "row_id"]).reset_index(drop=True),
    )


cv_detail, score_cv_detail, oof_predictions = run_stacked_xgb_cv()

display(cv_detail)
display(score_cv_detail.head(20))
display(oof_predictions.head(10))

,model,fold,threshold,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision,log_loss,brier_score,predicted_yes_count
0,raw_xgboost,1,0.5,0.937500,0.938278,0.980769,0.894737,0.935780,0.964593,0.960947,0.241206,0.065586,52
1,stacked_ridge_scores_xgboost,1,0.5,0.946429,0.947049,0.981132,0.912281,0.945455,0.981180,0.985086,0.184235,0.049775,53
2,oracle_true_scores_xgboost,1,0.5,0.946429,0.947049,0.981132,0.912281,0.945455,0.980223,0.984919,0.194685,0.052581,53
3,raw_xgboost,2,0.5,0.866071,0.866188,0.875000,0.859649,0.867257,0.935566,0.930610,0.323369,0.098904,56
4,stacked_ridge_scores_xgboost,2,0.5,0.857143,0.857097,0.859649,0.859649,0.859649,0.932695,0.928382,0.349815,0.104071,57
5,oracle_true_scores_xgboost,2,0.5,0.875000,0.874641,0.864407,0.894737,0.879310,0.938118,0.928138,0.327595,0.096432,59
6,raw_xgboost,3,0.5,0.848214,0.845785,0.815385,0.913793,0.861789,0.916986,0.925340,0.383871,0.115653,65
7,stacked_ridge_scores_xgboost,3,0.5,0.875000,0.873563,0.854839,0.913793,0.883333,0.938059,0.950743,0.325942,0.100610,62
8,oracle_true_scores_xgboost,3,0.5,0.866071,0.864304,0.841270,0.913793,0.876033,0.925287,0.927261,0.365433,0.106490,63
9,raw_xgboost,4,0.5,0.883929,0.882822,0.868852,0.913793,0.890756,0.911877,0.887263,0.369609,0.101157,61


,fold,split,target,mae,rmse,r2,spearman,within_0_5,within_1_0
0,1,train_inner_oof,role_title_preference_score,0.428194,0.740521,0.542878,0.740318,0.732143,0.868304
1,1,train_inner_oof,role_family_preference_score,0.279904,0.489932,0.781019,0.822406,0.816964,0.933036
2,1,train_inner_oof,seniority_profile_fit_score,0.230639,0.410571,0.933493,0.917486,0.863839,0.953125
3,1,train_inner_oof,location_profile_fit_score,0.279019,0.429748,0.857170,0.869837,0.816964,0.955357
4,1,train_inner_oof,work_mode_fit_score,0.263282,0.463602,0.770029,0.853851,0.843750,0.933036
5,1,train_inner_oof,experience_profile_fit_score,0.198990,0.353561,0.954040,0.918468,0.866071,0.973214
6,1,train_inner_oof,skills_profile_fit_score,0.301811,0.521945,0.811237,0.849579,0.825893,0.939732
7,1,train_inner_oof,salary_profile_fit_score,0.525973,0.719963,0.834171,0.910989,0.593750,0.875000
8,1,train_inner_oof,overall_preference_score,0.527533,0.661877,0.815523,0.844317,0.551339,0.908482
9,1,outer_validation,role_title_preference_score,0.313741,0.540614,0.788022,0.835802,0.767857,0.928571


,row_id,company_name,role_title,role_family,seniority,location,applied,source_type,fold,actual_applied,true_role_title_preference_score,pred_role_title_preference_score,true_role_family_preference_score,pred_role_family_preference_score,true_seniority_profile_fit_score,pred_seniority_profile_fit_score,true_location_profile_fit_score,pred_location_profile_fit_score,true_work_mode_fit_score,pred_work_mode_fit_score,true_experience_profile_fit_score,pred_experience_profile_fit_score,true_skills_profile_fit_score,pred_skills_profile_fit_score,true_salary_profile_fit_score,pred_salary_profile_fit_score,true_overall_preference_score,pred_overall_preference_score,raw_xgboost_probability,raw_xgboost_prediction,stacked_ridge_scores_xgboost_probability,stacked_ridge_scores_xgboost_prediction,oracle_true_scores_xgboost_probability,oracle_true_scores_xgboost_prediction
0,1,Nirmata,"Software Engineer Intern, India",Backend / Cloud,intern,"Bangalore, Hybrid",no,deterministic_scrape_candidate,1,0,5.0,4.193507,4.0,4.083565,5.0,4.963473,4.0,4.349691,5.0,5.000000,5.0,4.580470,2.0,2.880153,4.0,2.226935,3.0,4.065217,0.974255,1,0.823865,1,0.771850,1
1,6,Conga,"Associate Software Engineer, TS",Fullstack / Backend,entry_level,India - Bangalore,yes,deterministic_scrape_candidate,1,1,5.0,4.409652,5.0,5.000000,5.0,4.947861,5.0,4.244636,5.0,5.000000,5.0,4.374601,5.0,5.000000,3.0,2.886605,5.0,3.712214,0.987272,1,0.954305,1,0.987971,1
2,10,Karat,Software Engineer (SWE3) (India),Backend / Fullstack,mid,Remote India - Bangalore only,yes,deterministic_scrape_candidate,1,1,3.0,3.785810,5.0,4.480936,3.0,2.659443,4.0,4.673188,5.0,4.167002,1.0,1.962095,4.0,3.656905,4.0,3.402430,2.0,2.243536,0.569398,1,0.557894,1,0.513295,1
3,15,Addepar,"Software Engineer, Backend – AI Platform",Backend / AI Platform,mid,"Pune, India",yes,deterministic_scrape_candidate,1,1,5.0,3.223088,5.0,4.106825,3.0,2.498028,4.0,3.686211,5.0,4.812664,2.0,2.311052,4.0,3.076713,7.0,6.945600,3.0,2.364931,0.409045,0,0.292348,0,0.385191,0
4,20,ConnectWise,Sr. Software Engineer - AI,AI / Generative AI,senior,India,no,deterministic_scrape_candidate,1,0,2.0,2.562011,4.0,4.030292,2.0,2.331883,4.0,4.504148,5.0,5.000000,2.0,1.557255,4.0,3.427080,6.0,6.787125,1.0,1.421989,0.134619,0,0.097085,0,0.163854,0
5,35,Point72,"Software Engineer, Middle Office Systems - AI/ML",Backend / AI/ML,mid,India likely / global,no,deterministic_scrape_candidate,1,0,5.0,3.165390,4.0,4.565208,2.0,2.449250,4.0,4.539572,5.0,4.888080,1.0,2.040048,4.0,3.210688,5.0,4.488899,1.0,2.332967,0.314607,0,0.367075,0,0.280166,0
6,36,Sumo Logic,Senior Machine Learning Engineer,AI/ML / Observability,senior,"Bengaluru, Karnataka, India",no,deterministic_scrape_candidate,1,0,3.0,2.345698,3.0,3.525364,1.0,0.927775,4.0,4.418688,5.0,5.000000,0.0,0.000000,3.0,3.583621,7.0,7.241209,2.0,0.670822,0.007848,0,0.009587,0,0.005723,0
7,38,Twilio,Staff Machine Learning Engineer (L4),AI/ML / LLM,staff,Remote - India,no,deterministic_scrape_candidate,1,0,1.0,2.048523,4.0,2.998569,1.0,0.049063,4.0,4.425690,5.0,4.808356,0.0,0.000000,3.0,2.772359,8.0,8.000000,1.0,0.869454,0.005819,0,0.006115,0,0.006607,0
8,39,Agoda,"Staff Machine Learning Engineer (India based, relocation)",AI/ML / Data,staff,"India based, relocation supported",no,deterministic_scrape_candidate,1,0,2.0,1.336763,4.0,3.447396,1.0,0.000000,5.0,4.457315,5.0,5.000000,0.0,0.000000,4.0,3.077782,8.0,8.000000,0.0,0.453144,0.008423,0,0.008150,0,0.006631,0
9,42,AlphaSense,Staff Software Engineer,Backend / AI Product,staff,Remote - India,no,deterministic_scrape_candidate,1,0,1.0,2.119633,4.0,3.499239,0.0,0.509022,5.0,4.216651,5.0,4.594488,0.0,0.092882,3.0,2.817510,7.0,6.977276,1.0,1.394333,0.089632,0,0.114012,0,0.053967,0


## Step 8: Aggregate CV Results

In [22]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

cv_summary = (
    cv_detail
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)

flat_cv_summary = cv_summary.copy()
flat_cv_summary.columns = [f"{metric}_{stat}" for metric, stat in flat_cv_summary.columns]
flat_cv_summary = flat_cv_summary.reset_index().sort_values("f1_yes_mean", ascending=False)

display(cv_summary)
display(flat_cv_summary)

score_cv_summary = (
    score_cv_detail
    .groupby(["split", "target"])[["mae", "rmse", "r2", "spearman", "within_0_5", "within_1_0"]]
    .agg(["mean", "std"])
)
display(score_cv_summary)

accuracy           balanced_accuracy            \
                                 mean       std              mean       std   
model                                                                         
oracle_true_scores_xgboost     0.9000  0.033048          0.899076  0.033622   
raw_xgboost                    0.8875  0.034349          0.887011  0.035366   
stacked_ridge_scores_xgboost   0.8875  0.034349          0.887075  0.034806   

                             precision_yes           recall_yes            \
                                      mean       std       mean       std   
model                                                                       
oracle_true_scores_xgboost        0.889333  0.054074   0.923472  0.026784   
raw_xgboost                       0.890457  0.061184   0.895705  0.022114   
stacked_ridge_scores_xgboost      0.888757  0.052479   0.895765  0.024990   

                                f1_yes             roc_auc            \
                                  mean       std      mean       std   
model                                                                  
oracle_true_scores_xgboost    0.905284  0.029939  0.948164  0.024551   
raw_xgboost                   0.891986  0.029982  0.937695  0.023962   
stacked_ridge_scores_xgboost  0.891701  0.032176  0.945483  0.022711   

                             average_precision            log_loss            \
                                          mean       std      mean       std   
model                                                                          
oracle_true_scores_xgboost            0.945568  0.029666  0.291916  0.072848   
raw_xgboost                           0.933245  0.030749  0.313934  0.065675   
stacked_ridge_scores_xgboost          0.943630  0.032131  0.297330  0.070088   

                             brier_score           predicted_yes_count  \
                                    mean       std                mean   
model                                                                    
oracle_true_scores_xgboost      0.083854  0.022613                60.0   
raw_xgboost                     0.090913  0.020819                58.2   
stacked_ridge_scores_xgboost    0.086453  0.021852                58.2   

                                        
                                   std  
model                                   
oracle_true_scores_xgboost    4.242641  
raw_xgboost                   4.969909  
stacked_ridge_scores_xgboost  3.563706

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_yes_mean,precision_yes_std,recall_yes_mean,recall_yes_std,f1_yes_mean,f1_yes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,log_loss_mean,log_loss_std,brier_score_mean,brier_score_std,predicted_yes_count_mean,predicted_yes_count_std
0,oracle_true_scores_xgboost,0.9000,0.033048,0.899076,0.033622,0.889333,0.054074,0.923472,0.026784,0.905284,0.029939,0.948164,0.024551,0.945568,0.029666,0.291916,0.072848,0.083854,0.022613,60.0,4.242641
1,raw_xgboost,0.8875,0.034349,0.887011,0.035366,0.890457,0.061184,0.895705,0.022114,0.891986,0.029982,0.937695,0.023962,0.933245,0.030749,0.313934,0.065675,0.090913,0.020819,58.2,4.969909
2,stacked_ridge_scores_xgboost,0.8875,0.034349,0.887075,0.034806,0.888757,0.052479,0.895765,0.024990,0.891701,0.032176,0.945483,0.022711,0.943630,0.032131,0.297330,0.070088,0.086453,0.021852,58.2,3.563706


mae                rmse  \
                                                   mean       std      mean   
split            target                                                       
outer_validation experience_profile_fit_score  0.187860  0.036579  0.328147   
                 location_profile_fit_score    0.266994  0.031016  0.406961   
                 overall_preference_score      0.493095  0.029573  0.616594   
                 role_family_preference_score  0.247469  0.037804  0.457545   
                 role_title_preference_score   0.386130  0.059743  0.677770   
                 salary_profile_fit_score      0.483670  0.044192  0.659653   
                 seniority_profile_fit_score   0.209610  0.034929  0.385870   
                 skills_profile_fit_score      0.269351  0.023642  0.476940   
                 work_mode_fit_score           0.238224  0.037634  0.412213   
train_inner_oof  experience_profile_fit_score  0.203808  0.010660  0.358461   
                 location_profile_fit_score    0.283513  0.009369  0.432541   
                 overall_preference_score      0.515849  0.015397  0.648720   
                 role_family_preference_score  0.257689  0.017822  0.478158   
                 role_title_preference_score   0.387954  0.023531  0.680569   
                 salary_profile_fit_score      0.509844  0.019334  0.695993   
                 seniority_profile_fit_score   0.218568  0.013675  0.384327   
                 skills_profile_fit_score      0.287623  0.010649  0.500452   
                 work_mode_fit_score           0.250373  0.008330  0.444450   

                                                               r2            \
                                                    std      mean       std   
split            target                                                       
outer_validation experience_profile_fit_score  0.062225  0.958678  0.013353   
                 location_profile_fit_score    0.043247  0.854782  0.050797   
                 overall_preference_score      0.055849  0.836865  0.026524   
                 role_family_preference_score  0.075658  0.804032  0.056487   
                 role_title_preference_score   0.097738  0.616483  0.101806   
                 salary_profile_fit_score      0.057490  0.856558  0.018701   
                 seniority_profile_fit_score   0.070094  0.939129  0.016738   
                 skills_profile_fit_score      0.055932  0.839840  0.041081   
                 work_mode_fit_score           0.084037  0.797482  0.034833   
train_inner_oof  experience_profile_fit_score  0.012448  0.952168  0.003048   
                 location_profile_fit_score    0.012345  0.845936  0.016634   
                 overall_preference_score      0.025985  0.820945  0.013281   
                 role_family_preference_score  0.013550  0.793446  0.011460   
                 role_title_preference_score   0.040003  0.623759  0.047985   
                 salary_profile_fit_score      0.026703  0.842678  0.007638   
                 seniority_profile_fit_score   0.023661  0.940469  0.006118   
                 skills_profile_fit_score      0.018462  0.827502  0.013502   
                 work_mode_fit_score           0.015646  0.769267  0.013878   

                                               spearman           within_0_5  \
                                                   mean       std       mean   
split            target                                                        
outer_validation experience_profile_fit_score  0.917538  0.010940   0.878571   
                 location_profile_fit_score    0.861055  0.016642   0.817857   
                 overall_preference_score      0.854667  0.017754   0.567857   
                 role_family_preference_score  0.839406  0.024400   0.851786   
                 role_title_preference_score   0.779784  0.044586   0.741071   
                 salary_profile_fit_score      0.924523  0.017586   0.630357   
            

## Step 9: Threshold Review

In [23]:
threshold_frames = []
for model_name in [
    "raw_xgboost",
    "stacked_ridge_scores_xgboost",
    "oracle_true_scores_xgboost",
]:
    threshold_frames.append(threshold_table(
        oof_predictions["actual_applied"],
        oof_predictions[f"{model_name}_probability"],
        model_name,
    ))

threshold_review = pd.concat(threshold_frames, ignore_index=True)
display(threshold_review.sort_values(["model", "f1_yes"], ascending=[True, False]).groupby("model").head(5))

,model,threshold,predicted_yes_count,precision_yes,recall_yes,f1_yes,balanced_accuracy
32,oracle_true_scores_xgboost,0.50,300,0.886667,0.923611,0.904762,0.899306
33,oracle_true_scores_xgboost,0.55,292,0.897260,0.909722,0.903448,0.899714
31,oracle_true_scores_xgboost,0.45,305,0.875410,0.927083,0.900506,0.893689
29,oracle_true_scores_xgboost,0.35,315,0.857143,0.937500,0.895522,0.886029
30,oracle_true_scores_xgboost,0.40,311,0.861736,0.930556,0.894825,0.886234
4,raw_xgboost,0.40,305,0.872131,0.923611,0.897133,0.890114
5,raw_xgboost,0.45,298,0.879195,0.909722,0.894198,0.888685
3,raw_xgboost,0.35,311,0.858521,0.927083,0.891486,0.882659
6,raw_xgboost,0.50,291,0.886598,0.895833,0.891192,0.887255
7,raw_xgboost,0.55,281,0.896797,0.875000,0.885764,0.884191


## Step 10: Save Artifacts

In [24]:
cv_detail_path = OUTPUT_DIR / "jedi_xgboost_stacked_outer_cv_detail.csv"
cv_summary_path = OUTPUT_DIR / "jedi_xgboost_stacked_outer_cv_summary.csv"
score_cv_detail_path = OUTPUT_DIR / "jedi_xgboost_stacked_ridge_score_cv_detail.csv"
oof_predictions_path = OUTPUT_DIR / "jedi_xgboost_stacked_oof_predictions.csv"
threshold_review_path = OUTPUT_DIR / "jedi_xgboost_stacked_threshold_review.csv"

cv_detail.to_csv(cv_detail_path, index=False)
flat_cv_summary.to_csv(cv_summary_path, index=False)
score_cv_detail.to_csv(score_cv_detail_path, index=False)
oof_predictions.to_csv(oof_predictions_path, index=False)
threshold_review.to_csv(threshold_review_path, index=False)

saved = pd.DataFrame({
    "artifact": [
        "cv_detail",
        "cv_summary",
        "ridge_score_cv_detail",
        "oof_predictions",
        "threshold_review",
    ],
    "path": [
        str(cv_detail_path),
        str(cv_summary_path),
        str(score_cv_detail_path),
        str(oof_predictions_path),
        str(threshold_review_path),
    ],
})

display(saved)

,artifact,path
0,cv_detail,d:\ML\JEDI\data\jedi_xgboost_stacked_outer_cv_detail.csv
1,cv_summary,d:\ML\JEDI\data\jedi_xgboost_stacked_outer_cv_summary.csv
2,ridge_score_cv_detail,d:\ML\JEDI\data\jedi_xgboost_stacked_ridge_score_cv_detail.csv
3,oof_predictions,d:\ML\JEDI\data\jedi_xgboost_stacked_oof_predictions.csv
4,threshold_review,d:\ML\JEDI\data\jedi_xgboost_stacked_threshold_review.csv


## Step 11: Readout

In [25]:
best_model = flat_cv_summary.iloc[0]["model"]
best_f1 = flat_cv_summary.iloc[0]["f1_yes_mean"]
best_auc = flat_cv_summary.iloc[0]["roc_auc_mean"]

print(f"Best CV model by F1_yes: {best_model}")
print(f"F1_yes mean: {best_f1:.4f}")
print(f"ROC-AUC mean: {best_auc:.4f}")

display(
    oof_predictions
    .sort_values("stacked_ridge_scores_xgboost_probability", ascending=False)
    .head(25)
    [[
        "row_id",
        "company_name",
        "role_title",
        "actual_applied",
        "stacked_ridge_scores_xgboost_probability",
        "raw_xgboost_probability",
        "oracle_true_scores_xgboost_probability",
    ] + [f"pred_{c}" for c in SCORE_TARGETS]]
)

Best CV model by F1_yes: oracle_true_scores_xgboost
F1_yes mean: 0.9053
ROC-AUC mean: 0.9482


,row_id,company_name,role_title,actual_applied,stacked_ridge_scores_xgboost_probability,raw_xgboost_probability,oracle_true_scores_xgboost_probability,pred_role_title_preference_score,pred_role_family_preference_score,pred_seniority_profile_fit_score,pred_location_profile_fit_score,pred_work_mode_fit_score,pred_experience_profile_fit_score,pred_skills_profile_fit_score,pred_salary_profile_fit_score,pred_overall_preference_score
158,341,ShopBack,New Grad Software Engineer - Web Platform - Data Products,1,0.996899,0.993765,0.994668,5.000000,5.000000,4.945315,4.946829,4.963911,4.904546,4.971343,4.369671,4.527477
247,257,Weekday,AI Accelerated App Developer,1,0.996636,0.988355,0.995785,4.672820,4.816964,4.931100,3.998187,4.383899,4.932175,4.915746,2.910549,4.586953
497,379,WATI.io,New Grad Software Engineer - Web Platform - Growth Platform,1,0.996283,0.994040,0.994655,4.854850,4.966898,4.905578,5.000000,3.939763,4.987918,4.945187,4.442724,4.359150
349,191,Slikk,Backend Engineer,1,0.995405,0.989393,0.991758,4.792950,4.909376,5.000000,3.808622,3.998411,4.813001,4.106182,4.072543,4.607582
248,258,Weekday,Founding Engineer,1,0.995291,0.993821,0.995580,5.000000,4.992501,4.898730,4.121872,4.903650,4.712170,4.892888,3.409354,4.516813
296,491,Weekday AI,New Grad Software Engineer - Web Platform - Data Products,1,0.995024,0.994154,0.995787,5.000000,5.000000,4.985960,4.540452,4.950717,5.000000,5.000000,3.845655,4.658971
171,402,WATI.io,New Grad Software Engineer - Web Platform - Core Platform,1,0.995010,0.992024,0.992145,5.000000,5.000000,4.902469,4.478082,4.951953,4.834586,5.000000,4.201873,4.430510
404,478,Apna,Junior Full Stack Developer - SaaS Platform,1,0.994932,0.992452,0.992885,5.000000,5.000000,5.000000,4.141291,4.937793,4.962950,5.000000,4.609253,4.382317
169,389,GreenChoice,New Grad Software Engineer - Web Platform - SaaS Platform,1,0.994882,0.992138,0.995108,4.924351,5.000000,4.794254,4.178890,3.969641,4.887011,4.975588,4.414146,4.463314
302,522,Stealth IVF AI,New Grad Software Engineer - Web Platform - Product Team,1,0.994391,0.994956,0.996430,5.000000,5.000000,4.956984,4.976262,5.000000,5.000000,5.000000,3.980302,4.544505
